# Common Mistakes in AI-Generated Code

AI code assistants are powerful but make predictable mistakes. Here's what to watch for.

## 1. Mutable Default Arguments

In [ ]:
# BUG: Mutable default argument
def add_item_bad(item, items=[]):
    items.append(item)
    return items

print(add_item_bad(1))  # [1]
print(add_item_bad(2))  # [1, 2] - Bug! List persists!
print(add_item_bad(3))  # [1, 2, 3] - Getting worse!

In [ ]:
# FIX: Use None as default
def add_item_good(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items

print(add_item_good(1))  # [1]
print(add_item_good(2))  # [2] - Fresh list each time
print(add_item_good(3))  # [3] - Correct!

## 2. Not Handling None/Missing Values

In [ ]:
# BUG: Assumes data exists
def get_user_email_bad(user):
    return user['profile']['email']  # Crashes if missing!

user = {'name': 'Alice'}  # No profile!
try:
    email = get_user_email_bad(user)
except KeyError as e:
    print(f"KeyError: {e}")

In [ ]:
# FIX: Safe navigation
def get_user_email_good(user):
    profile = user.get('profile', {})
    return profile.get('email', 'no-email')

user = {'name': 'Alice'}
email = get_user_email_good(user)
print(f"Email: {email}")  # 'no-email' - safe!

## 3. Modifying List While Iterating

In [ ]:
# BUG: Modifying list during iteration
numbers = [1, 2, 3, 4, 5, 6]
for n in numbers:
    if n % 2 == 0:
        numbers.remove(n)  # Dangerous!

print(f"Result: {numbers}")  # [1, 3, 5, 6] - Missed 6!

In [ ]:
# FIX: Use list comprehension or iterate over copy
numbers = [1, 2, 3, 4, 5, 6]

# Option 1: List comprehension (best)
numbers = [n for n in numbers if n % 2 != 0]
print(f"Comprehension: {numbers}")

# Option 2: Iterate over copy
numbers = [1, 2, 3, 4, 5, 6]
for n in numbers[:]:
    if n % 2 == 0:
        numbers.remove(n)
print(f"Copy iteration: {numbers}")

## 4. Variable Scope Issues

In [ ]:
# BUG: UnboundLocalError
counter = 0

def increment_bad():
    counter += 1  # Error! Python sees assignment, thinks it's local
    return counter

try:
    increment_bad()
except UnboundLocalError as e:
    print(f"UnboundLocalError: {e}")

In [ ]:
# FIX: Use global or nonlocal
counter = 0

def increment_good():
    global counter
    counter += 1
    return counter

print(increment_good())  # 1
print(increment_good())  # 2

# Better: Avoid global state entirely
def increment_best(counter):
    return counter + 1

my_counter = 0
my_counter = increment_best(my_counter)
print(f"Better approach: {my_counter}")

## 5. Shallow vs Deep Copy

In [ ]:
# BUG: Shallow copy shares nested objects
original = {'name': 'Alice', 'scores': [90, 85, 95]}
copy = original.copy()  # Shallow copy!

copy['name'] = 'Bob'  # OK - different string
copy['scores'].append(100)  # Bug! Same list!

print(f"Original scores: {original['scores']}")  # [90, 85, 95, 100] - Modified!
print(f"Copy scores: {copy['scores']}")

In [ ]:
# FIX: Use deep copy for nested structures
import copy

original = {'name': 'Alice', 'scores': [90, 85, 95]}
deep_copy = copy.deepcopy(original)

deep_copy['scores'].append(100)

print(f"Original scores: {original['scores']}")  # [90, 85, 95] - Unchanged!
print(f"Copy scores: {deep_copy['scores']}")  # [90, 85, 95, 100]

## 6. String vs Integer Comparisons

In [ ]:
# BUG: Comparing string to int
user_input = "5"

if user_input == 5:
    print("Five!")
else:
    print("Not five")  # This runs - "5" != 5

In [ ]:
# FIX: Convert types explicitly
user_input = "5"

if int(user_input) == 5:
    print("Five!")  # Now works

# Or compare as strings
if user_input == "5":
    print("Five (string)!")

## 7. Incorrect Exception Handling

In [ ]:
# BUG: Bare except hides all errors
def process_bad(data):
    try:
        result = data['key'] / data['divisor']
        return result
    except:  # Catches EVERYTHING - even typos!
        return None

# This hides the real bug (typo in 'divisor')
data = {'key': 10, 'divsor': 2}  # Typo!
result = process_bad(data)
print(f"Result: {result}")  # None - but why?

In [ ]:
# FIX: Catch specific exceptions
def process_good(data):
    try:
        result = data['key'] / data['divisor']
        return result
    except KeyError as e:
        print(f"Missing key: {e}")
        return None
    except ZeroDivisionError:
        print("Cannot divide by zero")
        return None

data = {'key': 10, 'divsor': 2}  # Typo - now we see it!
result = process_good(data)

## 8. Off-by-One Errors

In [ ]:
# BUG: Common off-by-one mistakes
items = ['a', 'b', 'c', 'd', 'e']

# Wrong: Tries to access index 5 (doesn't exist)
# for i in range(1, len(items) + 1):
#     print(items[i])  # IndexError on last iteration

# Wrong: Misses last item
for i in range(len(items) - 1):
    print(items[i], end=' ')
print("- Missing 'e'!")

In [ ]:
# FIX: Use proper ranges or iterate directly
items = ['a', 'b', 'c', 'd', 'e']

# Best: Don't use indices at all
for item in items:
    print(item, end=' ')
print()

# If you need indices, use enumerate
for i, item in enumerate(items):
    print(f"{i}: {item}")

## 9. Incorrect Boolean Logic

In [ ]:
# BUG: Confusing 'and' vs 'or' in conditions
age = 25

# Wrong: This is ALWAYS False
# (age can't be both < 18 AND > 65)
if age < 18 and age > 65:
    print("Discount applies")
else:
    print("No discount")  # Always this

In [ ]:
# FIX: Use correct logic
age = 70

# Correct: Either under 18 OR over 65
if age < 18 or age > 65:
    print("Discount applies")
else:
    print("No discount")

## 10. Import and Path Issues

In [ ]:
# BUG: AI sometimes generates imports that don't exist
# from utils import helper  # Module might not exist!
# from sklearn.magic import MagicClassifier  # Made up!

# Common issues:
# 1. Wrong module name
# 2. Wrong function name
# 3. Old API that changed
# 4. Package not installed

# FIX: Always verify imports
try:
    from nonexistent import module
except ModuleNotFoundError as e:
    print(f"Import error: {e}")
    print("Check: 1) spelling, 2) package installed, 3) correct function name")

## Quick Reference: Common Mistakes

| Mistake | Fix |
|---------|-----|
| Mutable default arg | Use `None`, create in function |
| Missing key access | Use `.get()` with default |
| Modify list while iterating | Use comprehension or copy |
| Global variable issues | Use `global`/`nonlocal` or avoid |
| Shallow copy | Use `copy.deepcopy()` |
| Type mismatch | Convert explicitly |
| Bare except | Catch specific exceptions |
| Off-by-one | Use `enumerate()` |
| Wrong boolean | Double-check and/or logic |
| Bad imports | Verify module/function names |

## Next Up

Debugging techniques and tools.

Continue to: [Debugging Techniques](02-debugging-techniques.ipynb)